# 01 — Spectral Preprocessing

Demonstrate the key preprocessing steps applied to APOGEE spectra before they are fed into the CNN classifier:

1. **Bad pixel masking** — flag and interpolate over pixels with non-zero bitmask values
2. **Continuum normalization** — fit and divide out the pseudo-continuum so absorption lines are measured relative to unity
3. **Batch preparation** — stack all spectra into a uniform `(N, 8575)` matrix suitable for CNN input

By the end of this notebook we will have a clean spectra matrix saved to `data/spectra_matrix.npy`.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table

# Path setup so we can import from src/
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data import get_visit_spectra, load_allstar_catalog
from src.preprocessing import (
    prepare_spectrum,
    batch_prepare_spectra,
    continuum_normalize,
    mask_bad_pixels,
)
from config import PREPROCESS_CONFIG, FEATURE_CONFIG

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1. Load Example Spectra

Load a few stars from the labeled sample to demonstrate the preprocessing steps. We pick 3 confirmed binaries and 3 confirmed single stars, preferring high-SNR targets so the spectral features are clearly visible.

In [ ]:
# Load labeled sample
labeled = Table.read(os.path.join(project_root, "data", "labeled_sample.fits"))
print(f"Labeled sample: {len(labeled)} stars")
print(f"Columns: {labeled.colnames}")

# Select 3 binary and 3 single stars with highest SNR
binaries = labeled[labeled["label"] == 1]
singles = labeled[labeled["label"] == 0]

binaries_sorted = binaries[np.argsort(binaries["SNR"])[::-1]]
singles_sorted = singles[np.argsort(singles["SNR"])[::-1]]

example_binaries = binaries_sorted[:3]
example_singles = singles_sorted[:3]

print(f"\nSelected binaries (SNR = {example_binaries['SNR'].data}):")
for row in example_binaries:
    print(f"  {row['APOGEE_ID'].strip()}")

print(f"\nSelected singles (SNR = {example_singles['SNR'].data}):")
for row in example_singles:
    print(f"  {row['APOGEE_ID'].strip()}")

# Load visit spectra for each example star
example_spectra = {}
for row in list(example_binaries) + list(example_singles):
    apogee_id = row["APOGEE_ID"].strip()
    label = "binary" if row["label"] == 1 else "single"
    try:
        visit = get_visit_spectra(
            apogee_id,
            telescope=row.get("TELESCOPE", "apo25m"),
            field=row["FIELD"].strip() if "FIELD" in row.colnames else None,
        )
        example_spectra[apogee_id] = {"visit": visit, "label": label, "meta": row}
        print(f"  Loaded {apogee_id}: {visit['flux'].shape[0]} visits")
    except Exception as e:
        print(f"  Could not load {apogee_id}: {e}")

print(f"\nLoaded spectra for {len(example_spectra)} / 6 example stars.")

## 2. Bad Pixel Masking

APOGEE bitmasks flag pixels affected by persistence, cosmic rays, telluric contamination, and other issues. `mask_bad_pixels()` replaces these with interpolated values so they do not corrupt downstream analysis.

In [ ]:
# Demonstrate bad pixel masking on one example star
example_id = list(example_spectra.keys())[0] if example_spectra else None

if example_id is not None:
    visit = example_spectra[example_id]["visit"]
    flux_raw = visit["flux"][0]
    wavelength = visit["wavelength"]
    bitmask = visit["bitmask"][0] if visit["bitmask"] is not None else None

    # Identify bad pixels
    bad = bitmask > 0 if bitmask is not None else np.zeros_like(flux_raw, dtype=bool)
    print(f"Star: {example_id}")
    print(f"Total pixels: {len(flux_raw)}, flagged: {bad.sum()} ({100*bad.mean():.1f}%)")

    # Apply masking
    flux_cleaned = mask_bad_pixels(flux_raw, bitmask)

    # Zoom into a window where bad pixels are visible
    wl_min, wl_max = 15500, 15800
    mask_window = (wavelength >= wl_min) & (wavelength <= wl_max)

    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

    # Raw spectrum with bad pixels highlighted
    axes[0].plot(wavelength[mask_window], flux_raw[mask_window], "k-", lw=0.8, label="Raw flux")
    bad_in_window = mask_window & bad
    if bad_in_window.any():
        axes[0].scatter(
            wavelength[bad_in_window], flux_raw[bad_in_window],
            c="red", s=15, zorder=5, label=f"Flagged pixels (N={bad_in_window.sum()})",
        )
    axes[0].set_ylabel("Flux (ADU)")
    axes[0].set_title(f"Raw Spectrum — {example_id}")
    axes[0].legend(fontsize=9)

    # Cleaned spectrum
    axes[1].plot(wavelength[mask_window], flux_cleaned[mask_window], "k-", lw=0.8, label="After mask_bad_pixels()")
    axes[1].set_xlabel(r"Wavelength ($\AA$)")
    axes[1].set_ylabel("Flux (ADU)")
    axes[1].set_title("Cleaned Spectrum")
    axes[1].legend(fontsize=9)

    fig.tight_layout()
    plt.show()
else:
    print("No example spectra loaded — run previous cell with data access.")

## 3. Continuum Normalization

Fit a pseudo-continuum to the spectrum and divide it out. We compare two approaches: polynomial fitting and median-filter smoothing. The goal is a normalized spectrum where the continuum level is ~1.0 and absorption lines dip below unity.

In [ ]:
# Compare continuum normalization approaches
if example_id is not None and example_id in example_spectra:
    visit = example_spectra[example_id]["visit"]
    flux_raw = visit["flux"][0]
    wavelength = visit["wavelength"]
    bitmask = visit["bitmask"][0] if visit["bitmask"] is not None else None

    # Clean bad pixels first
    flux_cleaned = mask_bad_pixels(flux_raw, bitmask)

    # Polynomial continuum fit
    flux_norm_poly = continuum_normalize(flux_cleaned, wavelength, method="polynomial")

    # Median filter continuum fit
    flux_norm_medfilt = continuum_normalize(flux_cleaned, wavelength, method="median_filter")

    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

    # Panel 1: Raw spectrum with continuum fits overlaid
    axes[0].plot(wavelength, flux_cleaned, "k-", lw=0.5, alpha=0.7, label="Cleaned flux")
    # Reconstruct continua by dividing cleaned by normalized
    cont_poly = flux_cleaned / flux_norm_poly
    cont_medfilt = flux_cleaned / flux_norm_medfilt
    axes[0].plot(wavelength, cont_poly, "b-", lw=1.5, alpha=0.8, label="Polynomial continuum")
    axes[0].plot(wavelength, cont_medfilt, "r-", lw=1.5, alpha=0.8, label="Median-filter continuum")
    axes[0].set_ylabel("Flux (ADU)")
    axes[0].set_title(f"Continuum Fitting — {example_id}")
    axes[0].legend(fontsize=9)

    # Panel 2: Polynomial normalized
    axes[1].plot(wavelength, flux_norm_poly, "b-", lw=0.6)
    axes[1].axhline(1.0, color="gray", ls="--", alpha=0.5)
    axes[1].set_ylabel("Normalized Flux")
    axes[1].set_title("Polynomial Normalization")
    axes[1].set_ylim(0.4, 1.15)

    # Panel 3: Median-filter normalized
    axes[2].plot(wavelength, flux_norm_medfilt, "r-", lw=0.6)
    axes[2].axhline(1.0, color="gray", ls="--", alpha=0.5)
    axes[2].set_ylabel("Normalized Flux")
    axes[2].set_xlabel(r"Wavelength ($\AA$)")
    axes[2].set_title("Median-Filter Normalization")
    axes[2].set_ylim(0.4, 1.15)

    fig.tight_layout()
    fig.savefig(
        os.path.join(project_root, "figures", "continuum_normalization.png"),
        dpi=150, bbox_inches="tight",
    )
    plt.show()
    print("Saved figures/continuum_normalization.png")
else:
    print("No example spectra available.")

## 4. Single Star vs Binary: Visual Comparison

The key visual showing what the CNN needs to learn. We compare normalized spectra of a single star and a binary candidate at similar Teff/logg, zooming into diagnostic absorption lines where binarity manifests as line broadening, asymmetry, or double-lined profiles.

In [ ]:
# Side-by-side comparison of single vs binary normalized spectra
single_ids = [k for k, v in example_spectra.items() if v["label"] == "single"]
binary_ids = [k for k, v in example_spectra.items() if v["label"] == "binary"]

if single_ids and binary_ids:
    sid = single_ids[0]
    bid = binary_ids[0]

    # Preprocess both
    s_visit = example_spectra[sid]["visit"]
    b_visit = example_spectra[bid]["visit"]
    wl = s_visit["wavelength"]

    s_flux = prepare_spectrum(
        s_visit["flux"][0], wl,
        s_visit["bitmask"][0] if s_visit["bitmask"] is not None else None,
    )
    b_flux = prepare_spectrum(
        b_visit["flux"][0], wl,
        b_visit["bitmask"][0] if b_visit["bitmask"] is not None else None,
    )

    # Define zoom regions around key absorption lines
    zoom_regions = {
        r"Mg I $15750\,\AA$": (15720, 15780),
        r"Fe I $15210\,\AA$": (15190, 15240),
        r"Fe I $15650\,\AA$": (15620, 15680),
    }

    fig, axes = plt.subplots(len(zoom_regions), 2, figsize=(14, 3.5 * len(zoom_regions)), sharex="row")

    for i, (label, (wl_lo, wl_hi)) in enumerate(zoom_regions.items()):
        mask = (wl >= wl_lo) & (wl <= wl_hi)

        # Single star
        axes[i, 0].plot(wl[mask], s_flux[mask], "k-", lw=1.0)
        axes[i, 0].axhline(1.0, color="gray", ls="--", alpha=0.4)
        axes[i, 0].set_ylabel("Normalized Flux")
        if i == 0:
            axes[i, 0].set_title(f"Single Star ({sid})")
        axes[i, 0].annotate(label, xy=(0.02, 0.05), xycoords="axes fraction", fontsize=10)

        # Binary star
        axes[i, 1].plot(wl[mask], b_flux[mask], "C3-", lw=1.0)
        axes[i, 1].axhline(1.0, color="gray", ls="--", alpha=0.4)
        if i == 0:
            axes[i, 1].set_title(f"Binary Candidate ({bid})")
        axes[i, 1].annotate(label, xy=(0.02, 0.05), xycoords="axes fraction", fontsize=10)

        # Annotate differences on the binary panel
        axes[i, 1].annotate(
            "broader / asymmetric",
            xy=(0.98, 0.92), xycoords="axes fraction",
            ha="right", fontsize=8, color="C3", fontstyle="italic",
        )

    for ax in axes[-1, :]:
        ax.set_xlabel(r"Wavelength ($\AA$)")

    fig.suptitle("Single Star vs Binary: Key Absorption Lines", fontsize=14, y=1.01)
    fig.tight_layout()
    fig.savefig(
        os.path.join(project_root, "figures", "single_vs_binary_spectra.png"),
        dpi=150, bbox_inches="tight",
    )
    plt.show()
    print("Saved figures/single_vs_binary_spectra.png")
else:
    print("Need at least one single and one binary star loaded.")

## 5. Batch Processing

Prepare the full spectra matrix for CNN input. Each star's first-epoch visit spectrum is preprocessed with `prepare_spectrum()` (bad pixel masking + continuum normalization) and stacked into an `(N, 8575)` array.

**Memory estimate:** 200K stars x 8575 pixels x 4 bytes = ~6.4 GB in float32. For local development we process a small subset; the full run is designed for Fornax.

In [ ]:
# This cell is designed to run on Fornax with full data access.
# For local development, use a small subset.
#
# Full pipeline:
# 1. For each star in labeled_sample, load visit 0 (first epoch)
# 2. Apply prepare_spectrum() 
# 3. Stack into (N, 8575) matrix
# 4. Save to data/spectra_matrix.npy

# Estimate: 200K stars x 8575 pixels x 4 bytes = ~6.4 GB (float32)
# For local testing, subset to N_SUBSET stars

N_SUBSET = 1000  # increase on Fornax

from astropy.table import Table
labeled = Table.read(os.path.join(project_root, "data", "labeled_sample.fits"))
# Subsample for development
subset = labeled[np.random.RandomState(42).choice(len(labeled), min(N_SUBSET, len(labeled)), replace=False)]
print(f"Processing {len(subset)} spectra...")

# This would run the full batch — commented out until data is available
# spectra_matrix = np.zeros((len(subset), 8575), dtype=np.float32)
# for i, row in enumerate(tqdm(subset)):
#     try:
#         visit = get_visit_spectra(row["APOGEE_ID"].strip(), field=row["FIELD"].strip())
#         spectra_matrix[i] = prepare_spectrum(visit["flux"][0], visit["wavelength"], visit["bitmask"][0] if visit["bitmask"] is not None else None)
#     except Exception as e:
#         spectra_matrix[i] = np.nan
# np.save(os.path.join(project_root, "data", "spectra_matrix.npy"), spectra_matrix)
# np.save(os.path.join(project_root, "data", "spectra_labels.npy"), subset["label"])
print("Batch processing cell ready — uncomment and run on Fornax with full data access.")